<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/Copy_of_002_train_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 002 - Train Model

In this notebook, we'll finetune a causal language model to map healthcare database column names to a standardized list of target column names. This process helps normalize varying column names from different data sources into a consistent format.


## Setup

In [1]:
!pip install tsilva_notebook_utils==0.0.58 > /dev/null

Loading API tokens and authentication secrets required for accessing Hugging Face, Weights & Biases, and notification services.


In [2]:
from tsilva_notebook_utils.colab import load_secrets_into_env

load_secrets_into_env([
    'HF_TOKEN',
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

Setting up configuration parameters for the model training, including batch sizes, learning rates, optimizer settings, and precision options.


In [3]:
import os
import torch
from tsilva_notebook_utils.colab import notebook_id_from_title

def setup_config():
    #@markdown ## General Settings

    #@markdown Seed for reproducible results:
    seed = 42 #@param {type:"integer"}
    #@markdown Hugging Face dataset to use for training:
    hf_dataset_id = "tsilva/clinical-field-mappings" #@param {type:"string"}
    #@markdown Base Hugging Face model to fine-tune:
    hf_source_model_id = "distilbert/distilgpt2" #@param {type:"string"}
    #@markdown Target Hugging Face model ID for saving the fine-tuned model:
    hf_target_model_id = "tsilva/clinical-field-mapper" #@param {type:"string"}
    #@markdown Task type for model: "classification" or "causal_lm"
    task_type = "causal_lm"  #@param ["classification", "causal_lm"]

    #@markdown ---

    #@markdown ## Training Parameters

    #@markdown ### Batch Configuration
    #@markdown Batch size per device for training:
    per_device_train_batch_size = 512 #@param {type:"integer"}
    #@markdown Batch size per device for evaluation (overrides training size if set):
    per_device_eval_batch_size = 512 #@param {type:"integer"}
    #@markdown Steps to accumulate gradients (simulates larger batches):
    gradient_accumulation_steps = 1 #@param {type:"integer"}

    #@markdown ---

    #@markdown ### Training Duration
    #@markdown Number of epochs to train (higher = better results, longer time):
    num_train_epochs = 50 #@param {type:"integer"}

    #@markdown ---

    #@markdown ### Learning Rate Settings
    #@markdown Initial learning rate for the optimizer:
    learning_rate = 5e-4 #@param {type:"number"}
    #@markdown Type of learning rate scheduler:
    lr_scheduler_type = "cosine" #@param ["linear", "cosine", "cosine_with_restarts", "polynomial", "constant", "constant_with_warmup"]
    #@markdown Warmup ratio (fraction of steps to ramp up learning rate):
    warmup_ratio = 0.01 #@param {type:"number"}

    #@markdown ---

    #@markdown ### Optimizer Configuration
    #@markdown Optimizer algorithm to use:
    optim = "adamw_bnb_8bit" #@param ["adamw", "adamw_torch", "adamw_bnb_8bit", "adamw_8bit", "adafactor"]
    #@markdown Weight decay to prevent overfitting:
    weight_decay = 0.1 #@param {type:"number"}
    #@markdown Max gradient norm for clipping (0 = no clipping):
    max_grad_norm = 0 #@param {type:"number"}

    #@markdown ---

    #@markdown ### Precision Settings
    #@markdown Data type for training (mixed precision enabled unless fp32):
    fp_datatype = "fp16" #@param ["fp32", "fp16", "bf16", "tf32"]

    #@markdown ---

    #@markdown ### Early Stopping
    #@markdown Stop training early if no improvement occurs:
    enable_early_stopping = True #@param {type:"boolean"}
    #@markdown Epochs to wait for improvement before stopping:
    early_stopping_patience = 5 #@param {type:"integer"}
    #@markdown Minimum improvement required to reset patience (0 = any gain):
    early_stopping_threshold = 0.0 #@param {type:"number"}

    #@markdown ---

    #@markdown ## Advanced Options
    #@markdown Use gradient checkpointing to reduce memory usage (may slow training):
    gradient_checkpointing = False #@param {type:"boolean"}
    #@markdown Enable DeepSpeed for optimized training (requires ds_config.json):
    use_deepspeed = True #@param {type:"boolean"}
    #@markdown Export model as ONNX:
    export_onnx = True #@param {type:"boolean"}
    manual_test_on_end = True #@param {type:"boolean"}
    pytorch_compile = True #@param {type:"boolean"}

    #@markdown ---

    # Infer mixed precision based on fp_datatype
    mixed_precision_training = (fp_datatype != "fp32")
    fp16 = (fp_datatype == "fp16" and mixed_precision_training)
    bf16 = (fp_datatype == "bf16" and mixed_precision_training)
    tf32 = (fp_datatype == "tf32" and mixed_precision_training)

    # Set notebook ID in the environment
    os.environ["NOTEBOOK_ID"] = notebook_id_from_title()

    # Assemble configuration dictionary
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device("cpu")
    config = {
        "device" : device,
        "seed": seed,
        "hf_dataset_id": hf_dataset_id,
        "hf_source_model_id": hf_source_model_id,
        "hf_target_model_id": f"{hf_target_model_id}-{task_type}",
        "export_onnx" : export_onnx,
        "manual_test_on_end" : manual_test_on_end,
        "pytorch_compile" : pytorch_compile,
        "task_type" : task_type,
        "trainer": {
            "per_device_train_batch_size": per_device_train_batch_size,
            "per_device_eval_batch_size": per_device_eval_batch_size,
            "gradient_accumulation_steps": gradient_accumulation_steps,
            "num_train_epochs": num_train_epochs,
            "learning_rate": learning_rate,
            "lr_scheduler_type": lr_scheduler_type,
            "warmup_ratio": warmup_ratio,
            "optim": optim,
            "weight_decay": weight_decay,
            "max_grad_norm": max_grad_norm,
            "fp_datatype": fp_datatype,
            "mixed_precision_training": mixed_precision_training,
            "fp16": fp16,
            "bf16": bf16,
            "tf32": tf32,
            "fp16_full_eval": fp16,
            "bf16_full_eval": bf16,
            "enable_early_stopping": enable_early_stopping,
            "early_stopping_patience": early_stopping_patience,
            "early_stopping_threshold": early_stopping_threshold,
            "gradient_checkpointing": gradient_checkpointing,
            "use_deepspeed": use_deepspeed
        }
    }
    return config

# Initialize global config
CONFIG = setup_config()
CONFIG

{'device': device(type='cuda'),
 'seed': 42,
 'hf_dataset_id': 'tsilva/clinical-field-mappings',
 'hf_source_model_id': 'distilbert/distilgpt2',
 'hf_target_model_id': 'tsilva/clinical-field-mapper-causal_lm',
 'export_onnx': True,
 'manual_test_on_end': True,
 'pytorch_compile': True,
 'task_type': 'causal_lm',
 'trainer': {'per_device_train_batch_size': 512,
  'per_device_eval_batch_size': 512,
  'gradient_accumulation_steps': 1,
  'num_train_epochs': 50,
  'learning_rate': 0.0005,
  'lr_scheduler_type': 'cosine',
  'warmup_ratio': 0.01,
  'optim': 'adamw_bnb_8bit',
  'weight_decay': 0.1,
  'max_grad_norm': 0,
  'fp_datatype': 'fp16',
  'mixed_precision_training': True,
  'fp16': True,
  'bf16': False,
  'tf32': False,
  'fp16_full_eval': True,
  'bf16_full_eval': False,
  'enable_early_stopping': True,
  'early_stopping_patience': 5,
  'early_stopping_threshold': 0.0,
  'gradient_checkpointing': False,
  'use_deepspeed': True}}

Installing necessary Python libraries for data processing, model training, and experiment tracking. This includes transformers, datasets, and optimization libraries.


In [4]:
# Importing IPython's API to execute shell commands from within the notebook
from IPython import get_ipython

# Getting the current IPython shell
ipython = get_ipython()

# Extracting configuration values from CONFIG dictionary
device = CONFIG["device"]
use_deepspeed = CONFIG["trainer"]["use_deepspeed"]
export_onnx = CONFIG["export_onnx"]

# Initial list of shell commands to run (start by updating package lists)
commands = [
    "apt update"
]

# Initial list of Python packages to install via pip
pip_packages = [
    "pandas",
    "seaborn",
    "matplotlib",
    "scikit-learn",
    "datasets",                    # HuggingFace datasets library
    "evaluate",                    # Evaluation library from HuggingFace
    "accelerate",                  # Multi-GPU / mixed precision helper library
    "bitsandbytes",                # 8-bit optimizer library
    "wandb"                        # Weights & Biases for experiment tracking
]

# If DeepSpeed is enabled, add DeepSpeed-specific system and Python packages
if use_deepspeed:
    # Install MPI and AIO libraries required for distributed training
    commands.append('apt install -y libopenmpi-dev openmpi-bin libaio-dev')
    # Check MPI version
    commands.append('mpiexec --version')
    # Add DeepSpeed-related Python packages
    pip_packages.extend(["mpi4py", "transformers[deepspeed]", "ninja"])

# If exporting to ONNX is enabled, add ONNX-related dependencies
if export_onnx:
    # Clone the transformers.js repository (likely for export or usage)
    commands.append('git clone https://github.com/huggingface/transformers.js.git')
    # Add ONNX-related Python packages
    pip_packages.extend(["onnx", "onnxslim", "onnxruntime", "optimum[onnx]"])

# Convert the list of pip packages into a single install command
pip_packages_s = " ".join(pip_packages)
commands.append(f"pip install {pip_packages_s}")

# If using a CUDA device, check GPU status with nvidia-smi
if "cuda" in device.type: commands.append("nvidia-smi")

# If DeepSpeed is enabled, run the DeepSpeed report command
if use_deepspeed: commands.append('ds_report')

# Print out all commands that will be run, formatted for readability
print("####" * 20)
print("The following commands will be executed:")
print("\n".join([f"- {x}" for x in commands]))
print("####" * 20)

def cmd(commands):
  for command in commands:
      ipython.system(command)

# Execute each command in the notebook environment
cmd(commands)

################################################################################
The following commands will be executed:
- apt update
- apt install -y libopenmpi-dev openmpi-bin libaio-dev
- mpiexec --version
- git clone https://github.com/huggingface/transformers.js.git
- pip install pandas seaborn matplotlib scikit-learn datasets evaluate accelerate bitsandbytes wandb mpi4py transformers[deepspeed] ninja onnx onnxslim onnxruntime optimum[onnx]
- nvidia-smi
- ds_report
################################################################################
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://arc

Setting a global random seed to ensure reproducible results across runs. This guarantees that random operations like weight initialization will be consistent.


In [5]:
from transformers import set_seed
set_seed(CONFIG["seed"])

Authenticating with Hugging Face to access model repositories and enable model uploads after training.


In [6]:
from huggingface_hub import login
login()

Initializing Weights & Biases (W&B) for experiment tracking, which will log metrics, hyperparameters, and model performance throughout training.


In [7]:
from wandb import login
login()

wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Load Model

Load raw dataset to retrieve labels:

In [8]:
from datasets import load_dataset
raw_dataset = load_dataset(CONFIG["hf_dataset_id"])
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['source', 'target'],
        num_rows: 9126
    })
    validation: Dataset({
        features: ['source', 'target'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['source', 'target'],
        num_rows: 1014
    })
})

Create sorted list of labels:

In [9]:
labels = set()
for split in raw_dataset.keys(): labels.update(raw_dataset[split]["target"])
labels = sorted(labels)
labels

['a1at_status',
 'acq_score',
 'act_score',
 'adas_cog_score',
 'admission_date',
 'admission_id',
 'admission_reason',
 'admission_source',
 'adverse_event_flag',
 'adverse_event_management_end_date',
 'adverse_event_management_start_date',
 'adverse_event_management_type',
 'adverse_event_permanent_discontinuation_flag',
 'adverse_event_type',
 'age',
 'alcohol_status',
 'allergy_reaction_type',
 'allergy_severity',
 'ami_event_date',
 'ana_result_status',
 'anesthesia_type',
 'animal_verbal_fluency_score',
 'ann_arbor_stage',
 'anti_ccp_level',
 'anti_ccp_status',
 'anxiety_depression_flag',
 'asa_physical_status',
 'asdas_score',
 'asthma_flag',
 'atrial_fibrillation_flag',
 'autoimmune_condition_reported',
 'basdai_score',
 'basfi_score',
 'benzodiazepine_use_flag',
 'best_response_date',
 'beta2_microglobulin_level',
 'biopsy_results_note',
 'biopsy_type',
 'birth_date',
 'bmi',
 'bmi_category',
 'bone_marrow_involvement_flag',
 'bone_marrow_plasma_cell_percent',
 'breast_cancer_

Loading the base pre-trained model and its corresponding tokenizer from Hugging Face, which will be fine-tuned for our specific task.


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

task_type = CONFIG["task_type"]
model_name = CONFIG["hf_source_model_id"]
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if task_type == "causal_lm": model = AutoModelForCausalLM.from_pretrained(model_name)
else: model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(labels))
model

[2025-05-05 16:55:25,521] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

Verifying that the model's tokenizer has end-of-sequence (EOS) and padding tokens, which are essential for proper text generation and batch processing.


In [11]:
tokenizer.eos_token, tokenizer.eos_token_id, tokenizer.pad_token, tokenizer.pad_token_id

('<|endoftext|>', 50256, None, None)

Adding a padding token to the tokenizer if missing, which is necessary for processing batches of varying lengths efficiently during training.


In [12]:
if not tokenizer.pad_token:
    tokenizer.add_special_tokens({'pad_token': '<|pad|>'}) # Add padding token to tokenizer
    model.resize_token_embeddings(len(tokenizer))          # Resize model embeddings to account for the new token
    model.config.pad_token_id = tokenizer.pad_token_id     # Update the model's config

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Double-check that the model now has a pad token:


In [13]:
tokenizer.eos_token, tokenizer.eos_token_id, tokenizer.pad_token, tokenizer.pad_token_id

('<|endoftext|>', 50256, '<|pad|>', 50257)

## Prepare dataset

Loading the dataset containing source-target pairs of healthcare column names from Hugging Face, which will be used to train and evaluate our model.


In [14]:
from datasets import load_dataset
raw_dataset = load_dataset(CONFIG["hf_dataset_id"])
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['source', 'target'],
        num_rows: 9126
    })
    validation: Dataset({
        features: ['source', 'target'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['source', 'target'],
        num_rows: 1014
    })
})

Converting text data into token IDs that the model can process, combining source and target with appropriate separators and end tokens.


In [15]:
import multiprocessing

task_type = CONFIG['task_type']
if task_type == 'causal_lm':
    # Tokenize the raw dataset in parallel using all available CPU cores
    tokenized_dataset = raw_dataset.map(
        # Define a lambda function to process each batch of rows
        lambda rows: tokenizer(
            # Prepare the input strings by combining 'source' and 'target' with a separator and the end-of-sequence token
            [f"{source}|{target}{tokenizer.eos_token}" for source, target in zip(rows["source"], rows["target"])],
            padding=False,        # Do not apply padding here (might be done later during batching)
            truncation=False,     # Do not truncate sequences here (can be controlled globally later)
            return_tensors=None   # Return tokenized output as a plain Python list/dict (not as tensors)
        ),
        batched=True,  # Process data in batches to improve efficiency
        num_proc=multiprocessing.cpu_count(),  # Use all available CPU cores for parallel processing
        remove_columns=["source", "target"]  # Remove original 'source' and 'target' columns from the output dataset
    )
    # Filter the dataset to keep only the examples where the input contains the EOS token, in batched mode
    tokenized_dataset = tokenized_dataset.filter(
        lambda batch: [
            tokenizer.eos_token_id in input_ids for input_ids in batch["input_ids"]
        ],
        batched=True
    )
else:
    label2id = {label: idx for idx, label in enumerate(labels)}
    id2label = {v: k for k, v in label2id.items()}
    model.config.label2id = label2id
    model.config.id2label = id2label

    def preprocess(example):
        return {
            "input_ids": tokenizer(example["source"], truncation=True)["input_ids"],
            "label": label2id[example["target"]]
        }

    tokenized_dataset = raw_dataset.map(preprocess)

tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 9126
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1014
    })
})

Smoke test the tokenized dataset across all splits to make sure that its data is ok:

In [16]:
import numpy as np

# Function to sample and print tokenized dataset entries
def sample_and_print(tokenized_dataset, sample_size=10):
    # Iterate over each split in the dataset (e.g., train, test, validation)
    for split in tokenized_dataset:
        print(f"### {split} ###")

        # Ensure the sample size does not exceed the number of available rows in the split
        actual_sample_size = min(sample_size, len(tokenized_dataset[split]))

        # Randomly select a sample of rows from the split
        sample_rows = np.random.choice(tokenized_dataset[split], size=actual_sample_size, replace=False)

        # Iterate over the sampled rows
        for row in sample_rows:
            # Decode the input_ids back to readable text
            decoded_text = tokenizer.decode(row["input_ids"])

            # Print the length of input_ids and the decoded text
            print(len(row["input_ids"]), decoded_text)

sample_and_print(tokenized_dataset)

### train ###
9 erosionScore|erosion_score<|endoftext|>
15 fev1perc|fev1_percent_predicted<|endoftext|>
13 ESR_Observacao|esr_level<|endoftext|>
16 FL_EmoScore|fact_lym_emotional_score<|endoftext|>
9 treatmentIntent01|treatment_intent<|endoftext|>
15 discharge_meds1|discharge_medications_reported<|endoftext|>
10 EventAmi|ami_event_date<|endoftext|>
20 beta2_microglobulin_level|beta2_microglobulin_level<|endoftext|>
9 ClinRespons|clinical_response<|endoftext|>
11 TipoBiopsia|biopsy_type<|endoftext|>
### validation ###
10 ACQ_Score|acq_score<|endoftext|>
12 TrigLevel3|triglycerides_level<|endoftext|>
16 EAMFlag5|extra_articular_manifestation_flag<|endoftext|>
14 clinical_ecog_status|ecog_performance_status<|endoftext|>
16 flag_anxiety_depression|anxiety_depression_flag<|endoftext|>
15 TratamentoFalhou|treatment_inefficacy_flag<|endoftext|>
10 reAdFlag5|readmission_flag<|endoftext|>
22 ConcomitantCorticosteroidFlag|concomitant_corticosteroid_flag<|endoftext|>
13 statA1at_ind|a1at_status<|

Setting up a data collator that will handle dynamic padding of sequences to the same length within each batch, optimizing memory usage during training.


In [17]:
task_type = CONFIG['task_type']
if task_type == 'causal_lm':
    from transformers import DataCollatorForLanguageModeling
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,  # Specify the tokenizer to use for processing text
        mlm=False,            # Set to False because we are not using masked language modeling (MLM), for example in causal language modeling
        return_tensors="pt"   # Specify that the output should be PyTorch tensors
    )
else:
    from transformers import DataCollatorWithPadding
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

data_collator

DataCollatorForLanguageModeling(tokenizer=GPT2TokenizerFast(name_or_path='distilbert/distilgpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|pad|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	50257: AddedToken("<|pad|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), mlm=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, tf_experimental_compile=False, return_tensors='pt', seed=None)

Verify that the collator is padding correctly and that the EOS token is included in the attention mask:


In [18]:
def debug_random_batch(dataset, data_collator, tokenizer, sample_size=3, seed=42):
    # Sampling and collating inside the function
    np.random.seed(seed)
    sampled_rows = np.random.choice(dataset["train"], size=sample_size, replace=False)
    collated_data = data_collator(list(sampled_rows))

    print(f"Debugging {sample_size} Random Samples from Dataset ({len(dataset['train'])} total)\n{'=' * 50}")

    for idx, (input_ids, attention_mask, labels) in enumerate(
        zip(collated_data["input_ids"], collated_data["attention_mask"], collated_data["labels"]), 1
    ):
        def summarize(tensor):
            return f"Len: {len(tensor)}, Non-zero: {sum(tensor)}, Raw: {tensor.tolist()}"

        decoded_input = tokenizer.decode(input_ids, skip_special_tokens=True)
        #valid_labels = [t for t in labels if t >= 0]
        #decoded_labels = tokenizer.decode(valid_labels, skip_special_tokens=True) if valid_labels else "<empty>"

        print(f"\nSample {idx}\n{'-' * 40}")
        print(f"Input IDs:\n  {summarize(input_ids)}\n  Decoded: {decoded_input}")
        print(f"Attention Mask:\n  {summarize(attention_mask)}")
        #print(f"Labels:\n  {summarize(labels)}\n  Decoded (filtered): {labels}")
        #print(f"Validation:\n  IDs == Mask: {len(input_ids) == len(attention_mask)}, IDs == Labels: {len(input_ids) == len(labels)}")
        print("=" * 50)

#debug_random_batch(tokenized_dataset, data_collator, tokenizer, sample_size=3)

## Evaluate Baseline

Defining a utility function that generates predictions from the model for given input texts, handling tokenization and device placement automatically.


In [19]:
import torch
from torch.nn.functional import softmax

def predict(sources, max_new_tokens=50, device=None):
    if device is None: device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    if not isinstance(sources, list): sources = [sources]
    is_lm = CONFIG.get("task_type", "classification") == "causal_lm"

    inputs = tokenizer(
        [f"{x}|" if is_lm else x for x in sources],
        padding=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        if is_lm:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            return [tokenizer.decode(o, skip_special_tokens=True).strip() for o in outputs]
        else:
            logits = model(**inputs).logits
            probs = softmax(logits, dim=-1)
            return [[(labels[i], float(p[i])) for i in p.argsort(descending=True)] for p in probs]

output = predict("cardi@")
if CONFIG["task_type"] == "causal_lm":
    print("Generated:", output[0])
else:
    for label, score in output[0][:5]:
        print(f"{label}: {score*100:.2f}%")

Generated: cardi@|||||||||||||||||||||||||||||||||||||||||||||||||||


Let's evaluate the model before training, the goal is to get a better test set accuracy post-training:

In [20]:
from tqdm import tqdm
from collections import defaultdict
import torch

def _evaluate(split, batch_size=128, max_new_tokens=50):
    model.to(CONFIG["device"])
    model.eval()
    is_lm = CONFIG.get("task_type") == "causal_lm"
    stats = defaultdict(lambda: {"hits": 0, "total": 0, "misses": []})

    dataset = raw_dataset[split] if is_lm else tokenized_dataset[split]
    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Evaluating {split}"):
        batch = dataset[i:i+batch_size]

        if is_lm:
            preds = predict(batch["source"], max_new_tokens)
            targets = [f"{s}|{t}" for s, t in zip(batch["source"], batch["target"])]
            target_labels = batch["target"]
            pred_labels = preds
        else:
            batch = data_collator(batch)
            with torch.no_grad():
                logits = model(**{k: v.to(CONFIG["device"]) for k, v in batch.items() if k != "labels"}).logits
                preds = torch.argmax(logits, dim=-1).tolist()
            targets = batch["labels"].tolist()
            target_labels = [id2label[t] for t in targets]
            pred_labels = [id2label[t] for t in preds]

        for pred, target, pred_label, target_label in zip(preds, targets, pred_labels, target_labels):
            _match = pred == target
            stats[target_label]["hits"] += int(_match)
            stats[target_label]["total"] += 1
            if not _match: stats[target_label]["misses"].append(pred if is_lm else {"predicted": pred_label, "actual": target_label})

    total = sum(x["total"] for x in stats.values())
    accuracy = sum(x["hits"] for x in stats.values()) / total if total else 0
    return {"accuracy": accuracy, "stats": dict(stats)}

def evaluate(*args, **kwargs):
    orig_pad = tokenizer.padding_side
    tokenizer.padding_side = "left"
    try: return _evaluate(*args, **kwargs)
    finally: tokenizer.padding_side = orig_pad

def evaluate_splits():
    keys = raw_dataset.keys() if CONFIG.get("task_type") == "causal_lm" else tokenized_dataset.keys()
    return {split: evaluate(split) for split in keys}

# Run
eval_stats = evaluate_splits()
print("\n")
for split, result in eval_stats.items():
    print(f"{split} accuracy: {result['accuracy']*100:.2f}%")

Evaluating test: 100%|██████████| 8/8 [00:03<00:00,  2.43it/s]



train accuracy: 0.00%
validation accuracy: 0.00%
test accuracy: 0.00%


## Train Model

Configuring and executing the model training process with our defined parameters, which will fine-tune the base model on our healthcare column mapping dataset.


In [21]:
import json
from tsilva_notebook_utils.huggingface import print_trainer_summary
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

def setup_trainer():
    # Extract trainer config variables from global CONFIG dictionary
    trainer_config = CONFIG["trainer"]

    # Assign variables from config for easier reference
    per_device_train_batch_size = trainer_config["per_device_train_batch_size"]  # Batch size per device (GPU/CPU) during training
    per_device_eval_batch_size = trainer_config["per_device_eval_batch_size"]  # Batch size per device during evaluation
    gradient_accumulation_steps = trainer_config["gradient_accumulation_steps"]  # Number of steps to accumulate gradients before updating model weights
    num_train_epochs = trainer_config["num_train_epochs"]  # Total number of training epochs
    learning_rate = trainer_config["learning_rate"]  # Initial learning rate
    lr_scheduler_type = trainer_config["lr_scheduler_type"]  # Learning rate scheduler type (e.g., 'linear', 'cosine')
    warmup_ratio = trainer_config["warmup_ratio"]  # Fraction of total steps used for learning rate warmup
    optim = trainer_config["optim"]  # Optimizer to use (e.g., 'adamw_torch')
    weight_decay = trainer_config["weight_decay"]  # Weight decay for regularization
    max_grad_norm = trainer_config["max_grad_norm"]  # Maximum gradient norm for clipping (to prevent exploding gradients)
    fp_datatype = trainer_config["fp_datatype"]  # Floating point data type (not used directly here)
    mixed_precision_training = trainer_config["mixed_precision_training"]  # Flag for mixed precision training (not used directly here)
    fp16 = trainer_config["fp16"]  # Enable mixed precision fp16 training (useful for faster training and lower memory usage)
    bf16 = trainer_config["bf16"]  # Enable bf16 mixed precision (Brain Floating Point 16-bit, used for better performance on some GPUs)
    tf32 = trainer_config["tf32"]  # TensorFloat-32 enabled (faster matmul on NVIDIA Ampere GPUs)
    fp16_full_eval = trainer_config["fp16_full_eval"]  # Enable mixed precision evaluation with fp16
    bf16_full_eval = trainer_config["bf16_full_eval"]  # Enable mixed precision evaluation with bf16
    enable_early_stopping = trainer_config["enable_early_stopping"]  # Enable early stopping to prevent overfitting
    early_stopping_patience = trainer_config["early_stopping_patience"]  # Number of evaluations with no improvement to wait before stopping
    early_stopping_threshold = trainer_config["early_stopping_threshold"]  # Threshold for considering improvement
    gradient_checkpointing = trainer_config["gradient_checkpointing"]  # Enable gradient checkpointing to reduce memory usage
    use_deepspeed = trainer_config["use_deepspeed"]  # Enable DeepSpeed for distributed, optimized training

    # Prepare DeepSpeed configuration if enabled
    deepspeed_config_path = None
    if use_deepspeed:
        # Define DeepSpeed config dictionary
        ds_config = {
            "train_batch_size": "auto",  # Let DeepSpeed auto-calculate total batch size
            "train_micro_batch_size_per_gpu": "auto",  # Auto micro-batch size per GPU
            "gradient_accumulation_steps": gradient_accumulation_steps,  # Number of steps to accumulate before optimizer step
            "steps_per_print": 100,  # How often (steps) DeepSpeed prints logs
            "optimizer": {
                "type": "AdamW",  # Optimizer type
                "params": {
                    "lr": learning_rate,  # Learning rate
                    "betas": [0.9, 0.999],  # Adam betas for momentum terms
                    "eps": 1e-8,  # Epsilon value to prevent division by zero
                    "weight_decay": weight_decay  # Weight decay for regularization
                }
            },
            "scheduler": {
                "type": "WarmupLR",  # Learning rate scheduler type
                "params": {
                    "warmup_min_lr": 0,  # Starting LR after warmup
                    "warmup_max_lr": learning_rate,  # Target LR after warmup
                    "warmup_num_steps": "auto"  # Automatically set warmup steps
                }
            },
            "fp16": {  # Mixed precision training with FP16
                "enabled": fp16,  # Enable FP16 precision
                "loss_scale": 0,  # Dynamic loss scaling
                "loss_scale_window": 1000,  # Window for adjusting loss scale
                "initial_scale_power": 16,  # Initial loss scale power
                "hysteresis": 2,  # Hysteresis for loss scale adjustment
                "min_loss_scale": 1  # Minimum loss scale
            },
            "zero_optimization": {  # Zero Redundancy Optimizer (ZeRO) stage 2
                "stage": 2,  # ZeRO stage (2 = optimizer state partitioning + gradients partitioning)
                "allgather_partitions": True,  # Gather partitioned optimizer states
                "allgather_bucket_size": 2e8,  # Bucket size for all-gather operations
                "overlap_comm": True,  # Overlap communication and computation
                "reduce_scatter": True,  # Enable reduce-scatter optimization
                "reduce_bucket_size": 2e8,  # Bucket size for reduce operations
                "contiguous_gradients": True  # Allocate gradients contiguously in memory
            },
            "gradient_clipping": max_grad_norm,  # Clip gradients to prevent exploding gradients
            "wall_clock_breakdown": False,  # Whether to enable timing breakdown reports
            "zero_allow_untested_optimizer": True  # Allow custom optimizers not tested by DeepSpeed
        }

        # Save DeepSpeed config to JSON file
        deepspeed_config_path = "ds_config.json"
        with open(deepspeed_config_path, 'w') as f:
            json.dump(ds_config, f, indent=2)

    # Prepare HuggingFace TrainingArguments dictionary
    training_args = {
        "seed": CONFIG["seed"],  # Random seed for reproducibility
        "output_dir": "./results",  # Directory to save model checkpoints and outputs
        "overwrite_output_dir": True,  # Overwrite contents of output directory
        "num_train_epochs": num_train_epochs,  # Total number of training epochs
        "per_device_train_batch_size": per_device_train_batch_size,  # Batch size per device (train)
        "per_device_eval_batch_size": per_device_eval_batch_size,  # Batch size per device (eval)
        "gradient_accumulation_steps": gradient_accumulation_steps,  # Accumulate gradients over multiple steps
        "learning_rate": learning_rate,  # Initial learning rate
        "lr_scheduler_type": lr_scheduler_type,  # Scheduler type
        "warmup_ratio": warmup_ratio,  # Warmup steps as a fraction of total steps
        "optim": optim,  # Optimizer to use
        "weight_decay": weight_decay,  # Weight decay for regularization
        "max_grad_norm": max_grad_norm,  # Max norm for gradient clipping
        "fp16": fp16,  # Use mixed precision fp16
        "bf16": bf16,  # Use mixed precision bf16
        "tf32": tf32,  # Enable tf32 (for Tensor Cores)
        "fp16_full_eval": fp16_full_eval,
        "bf16_full_eval": bf16_full_eval,
        "eval_strategy": "epoch",  # Evaluation frequency ("epoch" = every epoch)
        "save_strategy": "epoch",  # Save model checkpoint frequency
        "save_total_limit": 3,  # Max number of saved checkpoints
        "load_best_model_at_end": enable_early_stopping,  # Reload best model at end of training if early stopping is enabled
        "metric_for_best_model": "eval_loss",  # Metric to choose the best model
        "greater_is_better": False,  # Lower metric value is better (for loss)
        "logging_strategy": "epoch",  # Logging frequency
        "logging_steps": 0,  # Number of steps between logging (not used here since logging is per epoch)
        "gradient_checkpointing": gradient_checkpointing,  # Enable gradient checkpointing to reduce memory
        "report_to": "wandb",  # Reporting destination (Weights and Biases)
        "push_to_hub": True,  # Push model to HuggingFace Hub after training
        "hub_model_id": CONFIG["hf_target_model_id"],  # HuggingFace Hub model ID
        "hub_strategy": "end",  # Push to hub strategy ("end" = after training)
        "deepspeed": deepspeed_config_path if use_deepspeed else None  # DeepSpeed config path if enabled
    }

    # Initialize callbacks list
    callbacks = []
    if enable_early_stopping:
        # Add early stopping callback to callbacks list
        callbacks.append(
            EarlyStoppingCallback(
                early_stopping_patience=early_stopping_patience,  # Number of bad evals to wait before stopping
                early_stopping_threshold=early_stopping_threshold  # Minimal change to be considered as an improvement
            )
        )

    # Initialize the Trainer
    trainer = Trainer(
        model=model,  # Model to be trained
        args=TrainingArguments(**training_args),  # Training arguments
        train_dataset=tokenized_dataset["train"],  # Training dataset
        eval_dataset=tokenized_dataset["validation"],  # Evaluation dataset
        processing_class=tokenizer,  # Tokenizer class
        callbacks=callbacks,  # Callbacks (e.g., early stopping)
        data_collator=data_collator  # Function to collate data batches
    )

    return trainer

# Set up the trainer by calling the setup function
trainer = setup_trainer()

# Train the model
trainer_result = trainer.train()

# Print a summary of training results
print_trainer_summary(trainer_result)

Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/fused_adam/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fused_adam...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
Loading extension module fused_adam...


Time to load fused_adam op: 0.033162832260131836 seconds
[2025-05-05 16:56:13,395] [WARNING] [lr_schedules.py:683:get_lr] Attempting to get learning rate from scheduler before it has started


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,4.639200,2.928846
2,2.356900,1.961892
3,1.710500,1.591313
4,1.419900,1.420258
5,1.246900,1.315940
6,1.123000,1.257229
7,1.014100,1.220517
8,0.929300,1.208214
9,0.850900,1.198097
10,0.787100,1.203149


Time: 172.70
Samples/second: 2642.17
total_memory_gb: 22.1610
allocated_memory_gb: 1.0904
cached_memory_gb: 20.0254
free_memory_gb: 21.0706


Evaluating the trained model on training, validation, and test splits to measure its performance and generalization capability on the column mapping task.


In [22]:
eval_stats = evaluate_splits()
print("\n")
for split, result in eval_stats.items():
    print(f"Accuracy for {split}: {result['accuracy'] * 100:.2f}%")

Evaluating test: 100%|██████████| 8/8 [00:00<00:00,  8.36it/s]



Accuracy for train: 98.24%
Accuracy for validation: 89.84%
Accuracy for test: 89.35%


Uploading the best-performing model checkpoint to the Hugging Face Hub, making it available for future use and sharing with the community.


In [23]:
trainer.push_to_hub(commit_message=f"Push {CONFIG['task_type']} fine-tuned model")

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/7.35k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/tsilva/clinical-field-mapper-causal_lm/commit/63992c0b82359ed2f7377fbe2b977e41faf1b295', commit_message='Push causal_lm fine-tuned model', commit_description='', oid='63992c0b82359ed2f7377fbe2b977e41faf1b295', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tsilva/clinical-field-mapper-causal_lm', endpoint='https://huggingface.co', repo_type='model', repo_id='tsilva/clinical-field-mapper-causal_lm'), pr_revision=None, pr_num=None)

Converting the model to ONNX format if specified in the config, which provides interoperability across different frameworks and optimized inference.


In [24]:
import os

def export_model_to_onnx():
    # Import the upload function from HuggingFace Hub
    from huggingface_hub import upload_file

    target_model_id = CONFIG['hf_target_model_id']
    onnx_repo_path = "onnx/model.onnx"
    local_onnx_path = f"/content/transformers.js/models/{target_model_id}/{onnx_repo_path}"

    # Ensure the local ONNX file does not exist before proceeding
    if os.path.exists(local_onnx_path): os.unlink(local_onnx_path)
    assert not os.path.exists(local_onnx_path), f"File still exists: {local_onnx_path}"

    # Step 1: Convert the model to ONNX format using an external script
    cmd([
        f"cd transformers.js; python -m scripts.convert --model_id '{target_model_id}'"
    ])

    # Step 2: Upload the generated ONNX file to the Hugging Face repository
    upload_file(
        path_or_fileobj=local_onnx_path,
        path_in_repo=onnx_repo_path,
        repo_id=target_model_id,
        commit_message=f"Add ONNX variant of {target_model_id}"
    )

# Step 3: Execute the export function if the flag is enabled
if export_onnx:
    export_model_to_onnx()

2025-05-05 16:59:47.996332: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746464388.018509   25755 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746464388.025207   25755 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[2025-05-05 16:59:51,376] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)
config.json: 100% 1.00k/1.00k [00:00<00:00, 6.86MB/s]
tokenizer_config.json: 100% 674/674 [00:00<00:00, 5.04MB/s]
vocab.json: 100% 798k/798k [00:00<00:00, 3.69MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 83.4MB/s]
tokenizer.json: 100% 3.56M/3.56M [00:00<00:00, 13.2MB/s]
added_tokens.json: 100% 23.0/23.0 [00:00<00:

model.onnx:   0%|          | 0.00/328M [00:00<?, ?B/s]

In [25]:
from huggingface_hub import HfApi

def push_model_card(eval_stats):
    hf_target_model_id = CONFIG["hf_target_model_id"]
    base_model = CONFIG["hf_source_model_id"]
    hf_dataset_id = CONFIG["hf_dataset_id"]

    api = HfApi()
    repo_info = api.repo_info(hf_dataset_id, repo_type="dataset")
    revision = repo_info.sha
    hf_dataset_url = f"https://huggingface.co/datasets/{hf_dataset_id}/tree/{revision}"

    accuracy_summary = "\n".join(
        [f"- **{split} accuracy**: {result['accuracy']*100:.2f}%" for split, result in eval_stats.items()]
    )

    # Prepare dynamic metadata
    metrics_metadata = [
        {
            "name": f"{split} Accuracy",
            "type": "accuracy",
            "value": result["accuracy"]
        }
        for split, result in eval_stats.items()
    ]

    final_train_loss = trainer_result.training_loss
    final_eval_result = trainer.evaluate()
    final_eval_loss = final_eval_result["eval_loss"]
    final_epoch = int(trainer.state.epoch)
    early_stopping_triggered = final_epoch < CONFIG["trainer"]["num_train_epochs"]

    task_name = "Text Classification" if CONFIG["task_type"] == "classification" else "Text Generation"
    task_type = "text-classification" if CONFIG["task_type"] == "classification" else "text-generation"

    # YAML front matter (dynamic)
    yaml_metadata = f"""---
library_name: transformers
license: apache-2.0
tags:
  - healthcare
  - column-normalization
  - causal-language-model
  - {base_model.split('/')[-1]}
model-index:
  - name: {hf_target_model_id}
    results:
      - task:
          name: {task_name}
          type: {task_type}
        dataset:
          name: {hf_dataset_id}
          type: healthcare
        metrics:
"""

    # Add metrics dynamically
    for metric in metrics_metadata:
        yaml_metadata += f"          - name: {metric['name']}\n"
        yaml_metadata += f"            type: {metric['type']}\n"
        yaml_metadata += f"            value: {metric['value']:.4f}\n"

    yaml_metadata += "---\n\n"

    model_card = f"""
{yaml_metadata}

# Model Card for {hf_target_model_id}

This model is a fine-tuned version of `{base_model}` on the [`{hf_dataset_id}`]({hf_dataset_url}) dataset.
Its purpose is to normalize healthcare database column names to a standardized set of target column names.

## Usage

This is a causal language model designed to map free-text field names to standardized schema terms.

Example:

```python
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("{hf_target_model_id}")
model = AutoModelForCausalLM.from_pretrained("{hf_target_model_id}")

def predict(input_text):
    inputs = tokenizer(input_text + "|", return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=50)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

predict('cardi@')
```

## Evaluation Results

{accuracy_summary}

## Training Details

- **Seed**: {CONFIG["seed"]}
- **Epochs scheduled**: {CONFIG["trainer"]["num_train_epochs"]}
- **Epochs completed**: {final_epoch}
- **Early stopping triggered**: {"Yes" if early_stopping_triggered else "No"}
- **Final training loss**: {final_train_loss:.4f}
- **Final evaluation loss**: {final_eval_loss:.4f}
- **Optimizer**: {CONFIG["trainer"]["optim"]}
- **Learning rate**: {CONFIG["trainer"]["learning_rate"]}
- **Batch size**: {CONFIG["trainer"]["per_device_train_batch_size"]}
- **Precision**: {CONFIG["trainer"]["fp_datatype"]}
- **DeepSpeed enabled**: {CONFIG["trainer"]["use_deepspeed"]}
- **Gradient accumulation steps**: {CONFIG["trainer"]["gradient_accumulation_steps"]}

## License

Specify your license here (e.g., Apache 2.0, MIT, etc.)

## Limitations and Bias

- Model was trained on a specific clinical mapping dataset.
- Performance may vary on out-of-distribution column names.
- Ensure you validate model outputs in production environments.
""".strip()

    with open("README.md", "w") as f:
        f.write(model_card.strip())

    # Push the new model card to the hub
    api.upload_file(
        path_or_fileobj="README.md",
        path_in_repo="README.md",
        repo_id=hf_target_model_id,
        commit_message="Update model card with evaluation results and training config."
    )

    report_file = "evaluation_report.json"
    with open(report_file, "w") as f:
        json.dump(eval_stats, f, indent=2)

    api.upload_file(
        path_or_fileobj=report_file,
        path_in_repo=report_file,
        repo_id=CONFIG["hf_target_model_id"],
        commit_message="Add detailed evaluation report."
    )

    print("✅ Model card pushed to hub!")

push_model_card(eval_stats)

✅ Model card pushed to hub!


Saving and uploading training artifacts including configuration files and environment details, ensuring the training process is fully reproducible.


In [26]:
def save_and_upload_artifacts(config, notebook_filename=None):
    import copy
    import json
    import subprocess
    import os
    import shutil
    from datetime import datetime
    from huggingface_hub import HfApi

    # === Setup ===
    timestamp = datetime.utcnow().strftime("%Y-%m-%dT%H-%M-%SZ")
    artifacts_folder = f"artifacts/{timestamp}"
    os.makedirs(artifacts_folder, exist_ok=True)

    # Inject timestamp into config for reproducibility
    config["run_timestamp"] = timestamp

    # Define paths
    config_path = os.path.join(artifacts_folder, "config.json")
    requirements_path = os.path.join(artifacts_folder, "requirements.txt")

    # === Save config.json ===
    with open(config_path, "w") as f:
        _config = copy.deepcopy(config)
        del _config["device"]
        json.dump(_config, f, indent=2)
    print(f"✅ Saved {config_path}")

    # === Save environment requirements.txt ===
    subprocess.run(["pip", "freeze"], stdout=open(requirements_path, "w"))
    print(f"✅ Saved {requirements_path}")

    # === Zip the artifacts folder ===
    zip_filename = f"{artifacts_folder}.zip"
    shutil.make_archive(artifacts_folder, 'zip', artifacts_folder)
    print(f"✅ Created ZIP archive: {zip_filename}")

    # === Upload to Hugging Face Hub ===
    api = HfApi()
    hf_target_model_id = config["hf_target_model_id"]

    # Upload individual files
    for file_name in os.listdir(artifacts_folder):
        file_path = os.path.join(artifacts_folder, file_name)
        api.upload_file(
            path_or_fileobj=file_path,
            path_in_repo=os.path.join(artifacts_folder, file_name),
            repo_id=hf_target_model_id,
            commit_message=f"Add {file_name} ({timestamp})"
        )
        print(f"✅ Uploaded {file_name}")

    # Upload ZIP file
    api.upload_file(
        path_or_fileobj=zip_filename,
        path_in_repo=os.path.join(artifacts_folder, os.path.basename(zip_filename)),
        repo_id=hf_target_model_id,
        commit_message=f"Add run artifact ZIP ({timestamp})"
    )
    print(f"✅ Uploaded ZIP file to {artifacts_folder}/")

    print("✅✅ All artifacts uploaded successfully!")

save_and_upload_artifacts(CONFIG, notebook_filename=os.getenv("NOTEBOOK_ID"))

✅ Saved artifacts/2025-05-05T17-01-24Z/config.json
✅ Saved artifacts/2025-05-05T17-01-24Z/requirements.txt
✅ Created ZIP archive: artifacts/2025-05-05T17-01-24Z.zip
✅ Uploaded config.json
✅ Uploaded requirements.txt


2025-05-05T17-01-24Z.zip:   0%|          | 0.00/6.28k [00:00<?, ?B/s]

✅ Uploaded ZIP file to artifacts/2025-05-05T17-01-24Z/
✅✅ All artifacts uploaded successfully!


Creating an interactive interface to manually test the trained model with custom inputs, helping to validate the model's behavior on specific examples.


In [27]:
def test_manually():
    while True:
        print("Type `exit` to stop...")
        source = input("Input: ")
        if source == 'exit': break
        preds = predict(source)[0]
        for label, prob in preds[:10]:
            print(f"{label}: {prob*100:.2f}%")
        #print(f"Prediction: {prediction}")
        if source in raw_dataset['train']['source']: print(f"⚠️ WARNING: `{source}` is part of training set.")

if CONFIG["manual_test_on_end"]: test_manually()

Type `exit` to stop...
Input: test


ValueError: not enough values to unpack (expected 2, got 1)

Set up automatic runtime disconnection after idle period to save resources:


In [ ]:
from tsilva_notebook_utils.colab import notify_and_disconnect_after_timeout
notify_and_disconnect_after_timeout()